[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [46]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    optimizer.zero_grad()
    print(type(micro_batches[0][0]))
    accumulated_loss=0
    for i, (x,y) in enumerate(micro_batches):
      loss=loss_fn(model(x),y)/len(micro_batches)
      print(loss)
      accumulated_loss+=loss
      #print(accumulated_loss)
      loss.backward()
    print(x,y)
    accumulated_loss/=i+1
    optimizer.step()
    print(i)
    return accumulated_loss.item()
    pass  # zero_grad, loop (forward, scale loss, backward), step

In [47]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

<class 'torch.Tensor'>
tensor(0.3790, grad_fn=<DivBackward0>)
tensor(0.8391, grad_fn=<DivBackward0>)
tensor(0.1574, grad_fn=<DivBackward0>)
tensor(0.0641, grad_fn=<DivBackward0>)
tensor([[ 1.0772, -0.9781, -1.6203, -0.6595],
        [ 0.9648,  0.0591, -0.1038,  0.8983]]) tensor([[-0.7463, -0.6661],
        [-0.5061,  0.2043]])
3
Loss: 0.3598864674568176


In [48]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
<class 'torch.Tensor'>
tensor(0.8133, grad_fn=<DivBackward0>)
tensor(0.3862, grad_fn=<DivBackward0>)
tensor([[ 0.3704,  1.4565,  0.9398,  0.7748],
        [ 0.1919,  1.2638, -1.2904, -0.7911]]) tensor([[-0.0209, -0.7185],
        [ 0.5186, -1.3125]])
1
  ✅ [1/3] Matches full batch update (7.9ms)
<class 'torch.Tensor'>
tensor(2.4771, grad_fn=<DivBackward0>)
tensor([[-1.7703,  0.2143, -0.5382,  0.5880],
        [ 1.6059,  0.4279, -0.6776,  1.0422]]) tensor([[-1.9513,  0.4186],
        [ 3.3214,  0.8764]])
0
  ✅ [2/3] Returns loss value (2.4ms)
<class 'torch.Tensor'>
tensor(4.1739, grad_fn=<DivBackward0>)
tensor([[ 0.1753, -0.9315, -1.5055, -0.6610],
        [ 1.3232,  0.0371, -0.2849, -0.1334]]) tensor([[ 1.8929,  3.1110],
        [-0.4584, -0.3360]])
0
  ✅ [3/3] Parameters actually update (2.3ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (12.6ms total)
  Progress 

# Notes
1. Implement this on pytorch lightning , deep speed,
2. Also think of faster version of implmentig this
3. What are the things optimizer.step()do , loss.backwards do etc